# Automated Sea Snake Contraction, Spline Generation, Straightening & Armature Rigging

This notebook demonstrates the end-to-end procedural animation pipeline:
1. **Contraction Skeleton Points**: Extract 1D skeleton points via mesh contraction & Algo B slice centroiding.
2. **Catmull-Rom Spline Generation**: Convert ordered skeleton points into a Catmull-Rom `Spline` object.
3. **Spine & Mesh Straightening**: Straighten the spline curve & 3D mesh using Bishop parallel transport frames.
4. **Straightened Armature Construction**: Build hierarchical `Armature` bones along the straightened spine.

In [1]:
import os
import sys
import time
import numpy as np
import torch
import trimesh
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

import bpy 
import numpy as np

sys.path.append("..")

from animgen.core.spline import Spline
from animgen.core.armature import Armature, Bone
from animgen.rigging.mesh_contraction import extract_skeleton
from animgen.rigging.refine_skelaton import refine_and_center_skeleton_iterative
from animgen.animation.straight import straighten, deform_mesh_to_spine_numpy

from animgen.io.glb_output import export_glb

print("Libraries successfully imported!")

Libraries successfully imported!


## 1. Load Sea Snake 3D Mesh

In [2]:
# 1. Load snake mesh
mesh_path = "../generated_data/test/dec_mesh_Sea_Snake.glb"
if not os.path.exists(mesh_path):
    mesh_path = "generated_data/test/dec_mesh_Sea_Snake.glb"
if not os.path.exists(mesh_path):
    mesh_path = "../generated_data/models/models_backup_3/dec_mesh_Sea_Snake.glb"
if not os.path.exists(mesh_path):
    mesh_path = "generated_data/models/models_backup_3/dec_mesh_Sea_Snake.glb"

scene = trimesh.load(mesh_path)
mesh = scene.to_mesh() if isinstance(scene, trimesh.Scene) else scene
print(f"Mesh loaded: {len(mesh.vertices)} vertices, {len(mesh.faces)} faces")

Mesh loaded: 6431 vertices, 12862 faces


## 2. Extract Skeleton Contraction Points & Algo B Refinement

In [3]:
t0 = time.time()
skel_v, skel_e = extract_skeleton(
    mesh, max_iters=20, threshold=0.5, no_1d_collapses=True, return_tuple=True
)
t_ext = time.time() - t0
print(f"[1] Contraction Skeleton Extracted in {t_ext:.2f}s: {len(skel_v)} nodes, {len(skel_e)} edges")

# High-Density Algo B Iterative Slice Centering (max_edge_len=0.1)
skel_v_final, skel_e_ref = refine_and_center_skeleton_iterative(mesh.vertices, skel_v, skel_e, max_edge_len=0.1, num_iters=10)
print(f"[2] High-Density Algo B Refined Skeleton: {len(skel_v_final)} nodes, {len(skel_e_ref)} edges")

[1] Contraction Skeleton Extracted in 7.15s: 18 nodes, 17 edges
[2] High-Density Algo B Refined Skeleton: 66 nodes, 65 edges


## 3. Convert Skeleton Points to Catmull-Rom Spline

In [4]:
# Trace ordered 1D spine chain
adj = {i: [] for i in range(len(skel_v_final))}
for u, v in skel_e_ref:
    adj[u].append(v)
    adj[v].append(u)

endpoints = [i for i, nbs in adj.items() if len(nbs) == 1]
start_node = endpoints[0] if len(endpoints) > 0 else 0

chain = [start_node]
visited = {start_node}
curr = start_node
while True:
    next_nodes = [nb for nb in adj[curr] if nb not in visited]
    if not next_nodes:
        break
    next_node = next_nodes[0]
    visited.add(next_node)
    chain.append(next_node)
    curr = next_node

ordered_verts = skel_v_final[chain]

# Convert points to Catmull-Rom Spline
pts_t_all = [torch.tensor(v, dtype=torch.float32) for v in ordered_verts]
spline = Spline(pts_t_all, alpha=1, phantom_num_points=1)
eval_pts_t = spline.evaluate_curve(num_points_per_segment=5)
source_spine = np.array([pt.detach().cpu().numpy() for pt in eval_pts_t])

print(f"../generated Catmull-Rom Spline Object from {len(ordered_verts)} control points ({len(source_spine)} evaluated curve points)")

../generated Catmull-Rom Spline Object from 64 control points (253 evaluated curve points)


## 4. Straighten Spline & Deform Mesh

In [5]:
# Straighten mesh using Spline curve
t0 = time.time()
straight_mesh = straighten(mesh, spine_points=spline, axis="x")
t_def = time.time() - t0
print(f"Straightened mesh along Spline in {t_def:.2f}s!")

# Compute target straight spine points matching cumulative arc length
seg_lens = np.linalg.norm(np.diff(source_spine, axis=0), axis=1)
s = np.concatenate(([0.0], np.cumsum(seg_lens)))
total_len = s[-1]

target_spine = np.zeros_like(source_spine)
target_spine[:, 0] = s
print(f"Straightened Spine Total Arc Length: {total_len:.4f} units")

Straightened mesh along Spline in 0.21s!
Straightened Spine Total Arc Length: 3.8295 units


## 5. Construct Hierarchical Armature on Straightened Spine & Export GLB

In [6]:
# Select 20 equidistant node points along straightened spine for bones
armature_node_indices = np.linspace(0, len(target_spine) - 1, 20, dtype=int)
straight_armature_verts = target_spine[armature_node_indices]

# Construct Armature
root_bone = Bone(
    head=tuple(straight_armature_verts[0]),
    tail=tuple(straight_armature_verts[1]),
)
straight_armature = Armature(root_bone)
curr_bone = root_bone
for i in range(2, len(straight_armature_verts)):
    curr_bone = straight_armature.add_connected_bone(
        curr_bone, tail=tuple(straight_armature_verts[i])
    )

print(f"Constructed Straightened Armature with {len(straight_armature.bones_list)} connected bones!")

out_dir = "../generated_data/rigged" if os.path.exists("../generated_data") else "generated_data/rigged"
os.makedirs(out_dir, exist_ok=True)
out_path_straight = os.path.join(out_dir, "straightened_dec_mesh_Sea_Snake.glb")
export_glb(straight_mesh, out_path_straight, armature=straight_armature)
print(f"Exported rigged & straightened snake mesh to: {out_path_straight}")

Constructed Straightened Armature with 19 connected bones!
Exported rigged & straightened snake mesh to: ../generated_data/rigged/straightened_dec_mesh_Sea_Snake.glb


## 6. Direct Straightening on Textured/Colored Snake (`paint_mesh_Sea_Snake.glb`)

`extract_skeleton` now automatically handles textured/unwelded meshes internally (by creating a manifold working copy). We can pass `paint_mesh` directly without any manual welding or pre-cleanup steps!

In [7]:
# 6.1 Load Colored Sea Snake Mesh
paint_mesh_path = "../generated_data/models/paint_mesh_Sea_Snake.glb"
if not os.path.exists(paint_mesh_path):
    paint_mesh_path = "generated_data/models/paint_mesh_Sea_Snake.glb"
if not os.path.exists(paint_mesh_path):
    paint_mesh_path = "../generated_data/models/models_backup_3/paint_mesh_Sea_Snake.glb"
if not os.path.exists(paint_mesh_path):
    paint_mesh_path = "generated_data/models/models_backup_3/paint_mesh_Sea_Snake.glb"

scene_paint = trimesh.load(paint_mesh_path)
paint_mesh = scene_paint.to_mesh() if isinstance(scene_paint, trimesh.Scene) else scene_paint
print(f"Loaded Textured Mesh: {paint_mesh_path}")
print(f"Vertices: {len(paint_mesh.vertices)}, Faces: {len(paint_mesh.faces)}")
print(f"Visual / Texture Type: {type(paint_mesh.visual)}")

Loaded Textured Mesh: ../generated_data/models/paint_mesh_Sea_Snake.glb
Vertices: 6993, Faces: 11126
Visual / Texture Type: <class 'trimesh.visual.texture.TextureVisuals'>


In [8]:
# 6.2 Extract Skeleton directly from paint_mesh (Internal Auto-Welding)
t0 = time.time()
paint_skel_v, paint_skel_e = extract_skeleton(
    paint_mesh, max_iters=20, threshold=0.5, no_1d_collapses=True, return_tuple=True
)
t_ext = time.time() - t0
print(f"[1] Contraction Skeleton Extracted directly from paint_mesh in {t_ext:.2f}s: {len(paint_skel_v)} nodes, {len(paint_skel_e)} edges")

# High-Density Iterative Slice Centering
paint_skel_v_final, paint_skel_e_ref = refine_and_center_skeleton_iterative(
    paint_mesh.vertices, paint_skel_v, paint_skel_e, max_edge_len=0.1, num_iters=10
)
print(f"[2] Refined Centered Skeleton: {len(paint_skel_v_final)} nodes, {len(paint_skel_e_ref)} edges")

[1] Contraction Skeleton Extracted directly from paint_mesh in 5.98s: 25 nodes, 25 edges
[2] Refined Centered Skeleton: 62 nodes, 62 edges


In [9]:
# 6.3 Trace 1D Spine Chain & Build Catmull-Rom Spline
adj = {i: [] for i in range(len(paint_skel_v_final))}
for u, v in paint_skel_e_ref:
    adj[u].append(v)
    adj[v].append(u)

endpoints = [i for i, nbs in adj.items() if len(nbs) == 1]
start_node = endpoints[0] if len(endpoints) > 0 else 0

chain = [start_node]
visited = {start_node}
curr = start_node
while True:
    next_nodes = [nb for nb in adj[curr] if nb not in visited]
    if not next_nodes:
        break
    next_node = next_nodes[0]
    visited.add(next_node)
    chain.append(next_node)
    curr = next_node

ordered_verts = paint_skel_v_final[chain]
pts_t = [torch.tensor(v, dtype=torch.float32) for v in ordered_verts]
paint_spline = Spline(pts_t, alpha=1, phantom_num_points=1)
eval_pts = paint_spline.evaluate_curve(num_points_per_segment=5)
source_spine = np.array([pt.detach().cpu().numpy() for pt in eval_pts])
print(f"Generated Clean Spline from {len(ordered_verts)} control points ({len(source_spine)} evaluated points)")

Generated Clean Spline from 59 control points (233 evaluated points)


In [10]:
# 6.4 Straighten Textured Mesh along Spline
t0 = time.time()
straight_paint_mesh = straighten(paint_mesh, spine_points=paint_spline, axis="x")
t_def = time.time() - t0
print(f"Straightened original textured mesh in {t_def:.2f}s! Vertices: {len(straight_paint_mesh.vertices)}")
print(f"Preserved Texture/Visuals: {type(straight_paint_mesh.visual)}")

# Compute target straight spine points matching cumulative arc length
seg_lens = np.linalg.norm(np.diff(source_spine, axis=0), axis=1)
s = np.concatenate(([0.0], np.cumsum(seg_lens)))
target_spine = np.zeros_like(source_spine)
target_spine[:, 0] = s
print(f"Straightened Spine Total Arc Length: {s[-1]:.4f} units")

Straightened original textured mesh in 0.22s! Vertices: 6993
Preserved Texture/Visuals: <class 'trimesh.visual.texture.TextureVisuals'>
Straightened Spine Total Arc Length: 3.7520 units


In [11]:
# 6.5 Construct Hierarchical Armature on Straightened Spine & Export Rigged Textured GLB
armature_indices = np.linspace(0, len(target_spine) - 1, 20, dtype=int)
straight_armature_verts = target_spine[armature_indices]

root_bone = Bone(
    head=tuple(straight_armature_verts[0]),
    tail=tuple(straight_armature_verts[1]),
)
paint_armature = Armature(root_bone)
curr_bone = root_bone
for i in range(2, len(straight_armature_verts)):
    curr_bone = paint_armature.add_connected_bone(
        curr_bone, tail=tuple(straight_armature_verts[i])
    )

print(f"Constructed Straightened Armature with {len(paint_armature.bones_list)} connected bones!")

out_dir = "../generated_data/rigged" if os.path.exists("../generated_data") else "generated_data/rigged"
os.makedirs(out_dir, exist_ok=True)
out_path_paint_straight = os.path.join(out_dir, "straightened_paint_mesh_Sea_Snake.glb")
export_glb(straight_paint_mesh, out_path_paint_straight, armature=paint_armature)
print(f"Successfully exported rigged & straightened textured snake to: {out_path_paint_straight}")

Constructed Straightened Armature with 19 connected bones!
Successfully exported rigged & straightened textured snake to: ../generated_data/rigged/straightened_paint_mesh_Sea_Snake.glb
